In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

dataset_path = "/content/drive/My Drive/GradProj"

# Check structure
for root, dirs, files in os.walk(dataset_path):
    print(f"Found directory: {root} with {len(files)} files")


Found directory: /content/drive/My Drive/GradProj with 0 files
Found directory: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco with 2 files
Found directory: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train with 2 files
Found directory: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/Sooty Falcon with 40 files
Found directory: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/steppe eagle with 30 files
Found directory: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/sand cat with 30 files
Found directory: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/Cape Hare with 15 files
Found directory: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/Rüppell's Vulture with 18 files
Found directory: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/Great knot with 24 files
Found directory: /content/drive/My Dri

In [ ]:
train_dir = "/content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train"
valid_dir = "/content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/valid"
test_dir = "/content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/test"


In [ ]:
import cv2
import numpy as np

def check_corrupt_images(directory):
    for root, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                img = cv2.imread(file_path)
                if img is None:
                    print(f"⚠️ Corrupt or unreadable image: {file_path}")
            except Exception as e:
                print(f"⚠️ Error reading {file_path}: {e}")

# Check train and valid directories
print("Checking train directory for corrupt images...")
check_corrupt_images(train_dir)

print("\nChecking valid directory for corrupt images...")
check_corrupt_images(valid_dir)


Checking train directory for corrupt images...

Checking valid directory for corrupt images...


In [ ]:
import os

file_path = "/content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/valid/Great knot/.DS_Store (1)"

if os.path.exists(file_path):
    os.remove(file_path)
    print(f"Deleted: {file_path}")
else:
    print("File not found.")


Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/valid/Great knot/.DS_Store (1)


In [ ]:
import os

def delete_unwanted_files(directory, extensions=[".json", ".DS_Store"]):
    for root, _, files in os.walk(directory):
        for file in files:
            if any(file.endswith(ext) for ext in extensions):
                file_path = os.path.join(root, file)
                try:
                    os.remove(file_path)
                    print(f"🗑️ Deleted: {file_path}")
                except Exception as e:
                    print(f"⚠️ Could not delete {file_path}: {e}")

# Clean train and valid directories
print("Cleaning train directory...")
delete_unwanted_files(train_dir)

print("\nCleaning valid directory...")
delete_unwanted_files(valid_dir)


Cleaning train directory...
🗑️ Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/_annotations.coco.json
🗑️ Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/.DS_Store
🗑️ Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/Arabian Gazelle/.DS_Store
🗑️ Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/saker falcon/.DS_Store
🗑️ Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/Black-legged Kittiwake/.DS_Store
🗑️ Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/egyptian vulture/.DS_Store
🗑️ Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/bateleur/.DS_Store
🗑️ Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/asir magpie/.DS_Store
🗑️ Deleted: /content/drive/My Drive/GradProj/ksa endangered animals.v3-v3.coco/train/Lappet-faced/.DS_Store
🗑

In [ ]:
import tensorflow as tf

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),
    batch_size=32
)

valid_dataset = tf.keras.utils.image_dataset_from_directory(
    valid_dir,
    image_size=(224, 224),
    batch_size=32
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=32
)


Found 489 files belonging to 21 classes.
Found 91 files belonging to 21 classes.
Found 33 files belonging to 21 classes.


In [ ]:
class_names = train_dataset.class_names
print("Classes:", class_names)


Classes: [' Rustic Bunting', 'Arabian Gazelle', 'Black-legged Kittiwake', 'Cape Hare', 'Common Pochard', 'European Turtle Dove', 'Great knot', 'Hawksbill turtles', 'Lappet-faced', 'Nubian Ibex', "Rüppell's Vulture", 'Sociable Lapwing', 'Sooty Falcon', 'asir magpie', 'bateleur', 'caracal', 'egyptian vulture', 'saker falcon', 'sand cat', 'steppe eagle', 'white headed duck']


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),  # Normalize pixel values
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')  # Output layer with softmax activation
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()  # Show model structure


/usr/local/lib/python3.11/dist-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)                │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 222, 222, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 111, 111, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 109, 109, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 54, 54, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 52, 52, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 26, 26, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 86528)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │      11,075,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 21)                  │           2,709 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 11,171,669 (42.62 MB)

 Trainable params: 11,171,669 (42.62 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs = 10  # Adjust based on dataset size

history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=epochs
)


Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 106s 6s/step - accuracy: 0.0492 - loss: 3.2242 - val_accuracy: 0.0549 - val_loss: 3.0845
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 104s 4s/step - accuracy: 0.1549 - loss: 2.8456 - val_accuracy: 0.0330 - val_loss: 3.2799
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 81s 4s/step - accuracy: 0.2485 - loss: 2.4800 - val_accuracy: 0.0769 - val_loss: 3.6741
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 64s 4s/step - accuracy: 0.4133 - loss: 2.0547 - val_accuracy: 0.1648 - val_loss: 5.0274
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 82s 4s/step - accuracy: 0.5705 - loss: 1.4976 - val_accuracy: 0.0989 - val_loss: 5.9343
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 82s 4s/step - accuracy: 0.7399 - loss: 0.8970 - val_accuracy: 0.0989 - val_loss: 8.7695
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 66s 4s/step - accuracy: 0.8385 - loss: 0.5618 - val_accuracy: 0.1868 - val_loss: 8.4341
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 81s 4s/step - accuracy: 0.9265 - loss: 0.2514 - val_accuracy: 0.2198 - val_los

In [ ]:
test_loss, test_acc = model.evaluate(test_dataset)
print("Test Accuracy:", test_acc)


2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 51ms/step - accuracy: 0.1531 - loss: 10.7079
Test Accuracy: 0.1515151560306549


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

# Load a test image
img_path = "/content/drive/MyDrive/GradProj/ksa endangered animals.v3-v3.coco/test/Black-legged Kittiwake/1200_jpg.rf.f308794baa66f58d42053f70c1e5ca71.jpg"  # Change path
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0  # Normalize

# Predict class
predictions = model.predict(img_array)
predicted_class = class_names[np.argmax(predictions)]
print("Predicted Class:", predicted_class)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
Predicted Class: Hawksbill turtles


In [ ]:
model.save("/content/drive/My Drive/GradProj/cnn_model.h5")
